# è fortemente consigliato runnare il notebook su colab

# Preliminari

In [1]:
%%capture
import os, re

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.55.4
!pip install --no-deps trl==0.22.2

In [2]:
import torch
import json
import copy
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset, Dataset
from unsloth import FastLanguageModel
from pathlib import Path
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq
from unsloth.chat_templates import train_on_responses_only
from unsloth.chat_templates import get_chat_template

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 2x faster
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # 4bit for 405b!
    "unsloth/Mistral-Small-Instruct-2409",     # Mistral 22b 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!

    "unsloth/Llama-3.2-1B-bnb-4bit",           # NEW! Llama 3.2 models
    "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    "unsloth/Llama-3.2-3B-bnb-4bit",
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",

    "unsloth/Llama-3.3-70B-Instruct-bnb-4bit" # NEW! Llama 3.3 70B!
] # More models at https://huggingface.co/unsloth

# Parametri

In [4]:
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.
model_name = "unsloth/Llama-3.2-3B-Instruct"
DATA_FILE_NAME = "data_training.json"
CHAT_TEMPLATE = "llama-3.1"

# Load model and tokenizer

In [5]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name, # or choose "unsloth/Llama-3.2-1B-Instruct"
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

==((====))==  Unsloth 2025.9.7: Fast Llama patching. Transformers: 4.55.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.9.7 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


# Data preparation

In [8]:
# Carichiamo il nostro file JSON
if "COLAB_" not in "".join(os.environ.keys()):
    data_path = Path('../data/' + DATA_FILE_NAME)
else:
    data_path = DATA_FILE_NAME

with open(data_path, "r") as f:
    raw_json_data = json.load(f)

I dati in `raw_json_data` sono salvati con la seguente struttura:
```
{
    "conversations": [
        {
            "messages": [
                {"role": "system", "content": <contenuto di sistema>},
                {"role": "user", "content": <contenuto di user>},
                {"role": "assistant", "content": <contenuto di assistant>}
            ]
        },
        ...
    ]
}
```

## Creazione dataset corretto

Trasformiamo `<contenuto di assistant>` in una stringa.

In [9]:
training_data = copy.deepcopy(raw_json_data)

for conversation in training_data['conversations']:
    for message in conversation['messages']:
        if message['role'] == 'assistant':
            message['content'] = json.dumps(message['content'])

Ora possiamo applicare il chat template, dato che il contenuto è una stringa

In [10]:
# Creiamo ora un "Huggingface dataset" con i nostri dati
dataset = Dataset.from_dict(training_data)
print(dataset)
display(dataset.features)

Dataset({
    features: ['conversations'],
    num_rows: 173
})


{'conversations': {'messages': [{'content': Value(dtype='string', id=None),
    'role': Value(dtype='string', id=None)}]}}

## Creazione nuova colonna

Noi vogliamo aggiungere la colonna "text" al nostro `dataset` che sia di questo tipo:
```
{
    "text": [
        <testo 1 formattato con chat template>,
        <testo 2 formattato con chat template>,
        ...
    ]
}
```
cioè popolato da stringhe formattate con il chat template del modello.

Dobbiamo passare dal generico formato di Huggingface `("role", "content")` al formato specifico del modello selezioanto.

In [11]:
# Creiamo la funzione da applicare successivamente al dataset per creare la nuova colonna "text" con i chat template corretti per il modello.
def formatting_prompts_func(data):
    conversations = data["conversations"]
    texts = [tokenizer.apply_chat_template(conversation['messages'], tokenize=False, add_generation_prompt=False) for conversation in conversations]
    return {"text" : texts}

In [12]:
# Applichiamo la funzione al datset
dataset = dataset.map(formatting_prompts_func, batched = True,)

# Ora abbiamo creato correttamente il nuovo campo "text"
print(dataset)
display(dataset.features)

Map:   0%|          | 0/173 [00:00<?, ? examples/s]

Dataset({
    features: ['conversations', 'text'],
    num_rows: 173
})


{'conversations': {'messages': [{'content': Value(dtype='string', id=None),
    'role': Value(dtype='string', id=None)}]},
 'text': Value(dtype='string', id=None)}

In [13]:
# Osserviamo ora il primo record del dataset
print(dataset[0]['text'])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 25 Sep 2025

Sei un modello di intelligenza artificiale specializzato nell'estrazione di informazioni cliniche dai referti radiologici.<|eot_id|><|start_header_id|>user<|end_header_id|>

RM ADDOME INFERIORE (S/C MDC)


L'ESAME E STATO ESEGUITO MEDINTE SEQUENZE FSE,DWI, MIRATO ALLA STADIAZIONE LOCALE DELLA NEOPLASIA RETTALE .

IN CORRISPONDENZA DELLA PARETE POSTEROLATERALE SINISTRA DEL RETTO MEDIO-BASSO SI SEGNALA FORMAZIONE AGGETTANTE CHE DA CIRCA 5 CM DALLO SFINTERE ANALE INTERNO SI ESTENDE CRANIALMENTE PER CIRCA 5 CM, OCCUPA 2/4 DELLE CIRCONFERENZA DEL LUME , INVIA DIGIRTAZIONI POLIPOIDI NEL MESORETTO POSTERIORE IN CONTINUITA CON DEPOSITI TUMORALI CHE AVVOLGONO A COLATA ED INGLOBANO I VASI EMORROIDARI SUPERIORI FINO AL PROMONTORIO SACRALE E RETRAGGONO LA FASCIA PERIRETTALE -PRESACRALE DALLE ORE 4 ALLE ORE 6.
ADESA ALLA FASCIA PERIRETTALE, ALLE ORE 7-8, E PRESENTE UNA LINFOAD

Vediamo come il chat template ha trasformato le conversazioni.

**[Notice]** Llama 3.1 Instruct's default chat template default adds `"Cutting Knowledge Date: December 2023\nToday Date: 26 July 2024"`, so do not be alarmed!

## Split data

In [14]:
splitted = dataset.train_test_split(test_size=0.2, seed=2025)
print(splitted)

DatasetDict({
    train: Dataset({
        features: ['conversations', 'text'],
        num_rows: 138
    })
    test: Dataset({
        features: ['conversations', 'text'],
        num_rows: 35
    })
})


In [15]:
train_split = splitted['train']
test_split = splitted['test']

<a name="Train"></a>
# Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support TRL's `DPOTrainer`!

In [16]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_split,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 60,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/138 [00:00<?, ? examples/s]

We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs.

In [17]:
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

Map (num_proc=2):   0%|          | 0/138 [00:00<?, ? examples/s]

We verify masking is actually done:

In [18]:
print(trainer.train_dataset[5].keys())
print(tokenizer.decode(trainer.train_dataset[5]["input_ids"]))
print('*--- inizio')
for x, y in zip(trainer.train_dataset[5]['input_ids'][:5], trainer.train_dataset[5]['labels'][:5]):
    print(f"id: {x:<10}label: {y}")
print('\n*--- fine')
for x, y in zip(trainer.train_dataset[5]['input_ids'][-5:], trainer.train_dataset[5]['labels'][-5:]):
    print(f"id: {x:<10}label: {y}")

dict_keys(['conversations', 'text', 'input_ids', 'attention_mask', 'labels'])
<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 25 Sep 2025

Sei un modello di intelligenza artificiale specializzato nell'estrazione di informazioni cliniche dai referti radiologici.<|eot_id|><|start_header_id|>user<|end_header_id|>

IN CORRISPONDENZA DELLA PARETE ANTERIORE DEL RETTO MEDIO-ALTO, DA CIRCA 7 CM DALLO SFINTERE ANALE INTERNO CON ESTENSIONE CRANIALE PER CIRCA 2 CM, SI DOCUMENTA ISPESSIMENTO PARIETALE CONFINATO ALLA PARETE (SPESSORE MASSIMO 13 MM), CHE OCCUPA DUE QUARTI DELLA CIRCONFERENZA DEL LUME, MODERATAMENTE IPERINTENSO NELLE SEQUENZE T2-DIPENDENTI ED IPERINTENSO NELLE SEQUENZE DWI, CHE MOSTRA INTENSA IMPREGNAZIONE CONSTRASTOGRAFICA NELLE SEQUENZE DI PERFUSIONE. 
DUE LINFONODI CON ASSE CORTO <5 MM IN SEDE MESORETTALE POSTERIORE, ASPECIFICI. NON EVIDENTI LINFOADENOPATIE IN SEDE PELVICA. 
PICCOLE CISTI DI TARLOV A L

In [19]:
space = tokenizer(" ", add_special_tokens = False).input_ids[0]  # id di sapce
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[5]["labels"]])

'                                                                                                                                                                                                                                                                                                                                 {"morfologia": "mucinoso", "spessore_parietale": 13.0, "estensione_cranio_caudale": 20.0, "distanza_oai": 70.0, "posizione": "medio", "carcinosi_peritoneale": "no", "lesioni_ossee": "no", "riflessione_peritoneale_anteriore": null, "infiltrazione_tessuto_adiposo": "no", "infiltrazione_sfinteri": "no", "infiltrazione_organi_extra": "no", "infiltrazione_organi_dettagli": null, "coinvolgimento_riflessione_peritoneale": "no", "coinvolgimento_fascia_mesorettale": "no", "linfonodi_sospetti": 0.0, "sedi_locoregionali": [], "sedi_non_locoregionali": [], "depositi_tumorali": "no", "numero_depositi": 0.0, "emvi_esteso": "no"}<|eot_id|>'

We can see the System and Instruction prompts are successfully masked!

In [20]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
3.07 GB of memory reserved.


In [21]:
# @title Training
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 138 | Num Epochs = 4 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,2.288300
2,2.268200
3,2.318500
4,2.180300
5,2.017600
6,1.814400
7,1.580600
8,1.399200
9,1.185400
10,1.007900


In [22]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

449.8785 seconds used for training.
7.5 minutes used for training.
Peak reserved memory = 4.656 GB.
Peak reserved memory for training = 1.586 GB.
Peak reserved memory % of max memory = 31.585 %.
Peak reserved memory for training % of max memory = 10.759 %.


<a name="Inference"></a>
# Inference
Let's run the model! You can change the instruction and input - leave the output blank!



We use `min_p = 0.1` and `temperature = 1.5`. Read this [Tweet](https://x.com/menhguin/status/1826132708508213629) for more information on why.

In [40]:
test_messages = test_split[0]['conversations']['messages'][:2]
tokenized_chat_template = tokenizer.apply_chat_template(
                        test_messages,
                        tokenize = True,
                        add_generation_prompt = True, # Must add for generation
                        return_tensors = "pt"
                        ).to("cuda")

In [42]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

outputs = model.generate(input_ids=tokenized_chat_template,
                         use_cache=True,
                         temperature=1.5,
                         min_p=0.1)

['<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\nSei un modello di intelligenza artificiale specializzato nell\'estrazione di informazioni cliniche dai referti radiologici.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nNEL RETTO MEDIO SI RILEVA VOLUMINOSA NEOFORMAZIONE POLIPOIDE VEGETANTE CON LARGA BASE D\'IMPIANTO LOCALIZZATA IN CORRISPONDENZA DELLA PARETE LATERALE DESTRA, DOVE SI RILEVA PARZIALE RETRAZIONE DELLE STRUTTURE FIBROVASCOLARI E DEL TESSUTO ADIPOSO MESORETTALE CIRCOSTANTE; LA NEOFORMAZIONE PRESENTA SUPERFICIE VILLOSA IRREGOLARE, MOSTRA DIMENSIONI ASSIALI MASSIME DI CIRCA 9,5 X 7,2 CM E LONGITUDINALI DI CIRCA 10 CM CON ESTENSIONE AL RETTO BASSO ED ALTO, DETERMINA OBLITERAZIONE PRESSOCHÈ COMPLETA DEL LUME.\r\nTALE LESIONE A LIVELLO DELLA BASE DI IMPIANTO - LUNGO LA PARETE LATERALE DESTRA - ASSUME ASPETTO RETRAENTE E MOSTRA FOCALE ESTENSIONE EXTRAPARIETALE (SE5IMA25); IN CORRISPONDENZA DE

In [102]:
char_lenght_inputs = len(tokenizer.decode(tokenized_chat_template[0]))
output_string = tokenizer.batch_decode(outputs)[0][char_lenght_inputs:-10]  # togliere <|eot_id|>
output_string_dict = json.loads(output_string)
display(output_string_dict)

{'morfologia': 'mucinoso',
 'spessore_parietale': None,
 'estensione_cranio_caudale': 10.0,
 'distanza_oai': None,
 'posizione': 'medio',
 'carcinosi_peritoneale': 'no',
 'lesioni_ossee': 'no',
 'riflessione_peritoneale_anteriore': 'sotto',
 'infiltrazione_tessuto_adiposo': 'sospetto',
 'infiltrazione_sfinteri': 'no',
 'infiltrazione_organi_extra': 'cavallo',
 'infiltrazione_organi_dettagli': None,
 'coinvolgimento_riflessione_peritoneale': 'no',
 'coinvolgimento_fascia_mesorettale': 'sospetto',
 'linfonodi_sospetti': 3.0,
 'sedi_locoregionali': ['mesorettali'],
 'sedi_non_locoregionali': [],
 'depositi_tumorali': 'no',
 'numero_depositi': 0.0,
 'emvi_esteso': 'sospetto'}

In [104]:
groundtruth = test_split[0]['conversations']['messages'][2]['content']
display(json.loads(groundtruth))

{'morfologia': 'solido_polipoide',
 'spessore_parietale': None,
 'estensione_cranio_caudale': 100.0,
 'distanza_oai': None,
 'posizione': 'medio',
 'carcinosi_peritoneale': 'no',
 'lesioni_ossee': 'no',
 'riflessione_peritoneale_anteriore': 'cavallo',
 'infiltrazione_tessuto_adiposo': 'si_5mm_plus',
 'infiltrazione_sfinteri': 'no',
 'infiltrazione_organi_extra': 'no',
 'infiltrazione_organi_dettagli': None,
 'coinvolgimento_riflessione_peritoneale': None,
 'coinvolgimento_fascia_mesorettale': 'rischio',
 'linfonodi_sospetti': 0.0,
 'sedi_locoregionali': ['mesorettali', 'rettali_superiori', 'otturatori'],
 'sedi_non_locoregionali': ['iliaci_esterni'],
 'depositi_tumorali': 'no',
 'numero_depositi': 0.0,
 'emvi_esteso': 'sospetto'}

 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [105]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference


from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids=tokenized_chat_template,
                   streamer=text_streamer,
                   use_cache=True,
                   temperature=1.5,
                   min_p=0.1)

{"morfologia": "solido_polipoide", "spessore_parietale": null, "estensione_cranio_caudale": 10.0, "distanza_oai": 72.0, "posizione": "medio", "carcinosi_peritoneale": "no", "lesioni_ossee": "no", "riflessione_peritoneale_anteriore": "basso", "infiltrazione_tessuto_adiposo": "si_5mm", "infiltrazione_sfinteri": "no", "infiltrazione_organi_extra": "no", "infiltrazione_organi_dettagli": null, "coinvolgimento_riflessione_peritoneale": "no", "coinvolgimento_fascia_mesorettale": "no", "linfonodi_sospetti": 10.0, "sedi_locoregionali": ["mesorettali", "otturatori"], "sedi_non_locoregionali": [], "depositi_tumorali": null, "numero_depositi": 0.0, "emvi_esteso": "no"}<|eot_id|>


<a name="Save"></a>
# Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [107]:
model.save_pretrained("lora_model")  # Local saving
tokenizer.save_pretrained("lora_model")
# model.push_to_hub("your_name/lora_model", token = "...") # Online saving
# tokenizer.push_to_hub("your_name/lora_model", token = "...") # Online saving

('lora_model/tokenizer_config.json',
 'lora_model/special_tokens_map.json',
 'lora_model/chat_template.jinja',
 'lora_model/tokenizer.json')

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [109]:
if True:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

==((====))==  Unsloth 2025.9.7: Fast Llama patching. Transformers: 4.55.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
{"morfologia": "mucinoso", "spessore_parietale": null, "estensione_cranio_caudale": 9.5, "distanza_oai": 6.67, "posizione": "medio", "carcinosi_peritoneale": null, "lesioni_ossee": null, "riflessione_peritoneale_anteriore": "sotto", "infiltrazione_tessuto_adiposo": "si", "infiltrazione_sfinteri": "si", "infiltrazione_organi_extra": "no", "infiltrazione_organi_dettagli": null, "coinvolgimento_riflessione_peritoneale": "si", "coinvolgimento_fascia_mesorettale": "si", "linfonodi_sospetti": 5.0, "sedi_locoregionali": ["mesorettali", "otturato

You can also use Hugging Face's `AutoModelForPeftCausalLM`. Only use this if you do not have `unsloth` installed. It can be hopelessly slow, since `4bit` model downloading is not supported, and Unsloth's **inference is 2x faster**.

# Alternative di salvataggio e caricamento

In [ ]:
if False:
    # I highly do NOT suggest - use Unsloth if possible
    from peft import AutoPeftModelForCausalLM
    from transformers import AutoTokenizer
    model = AutoPeftModelForCausalLM.from_pretrained(
        "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        load_in_4bit = load_in_4bit,
    )
    tokenizer = AutoTokenizer.from_pretrained("lora_model")

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False:
    model.save_pretrained("model")
    tokenizer.save_pretrained("model")
if False:
    model.push_to_hub("hf/model", token = "")
    tokenizer.push_to_hub("hf/model", token = "")


### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("model", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "hf/model", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "", # Get a token at https://huggingface.co/settings/tokens
    )

Now, use the `model-unsloth.gguf` file or `model-unsloth-Q4_K_M.gguf` file in llama.cpp.

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>
